# 01b — GBM miRNA expression (optional)

Builds a miRNA expression matrix from the GDC miRNA-seq files. The last section of notebook 02 reads `gbm_mirna_log2rpm.csv` to find miRNA regulators of regulons (mutation → miRNA → regulon). A copy of that file from the May 2026 run is already in the data folder, so this notebook only needs rerunning if the GDC downloads change.

Input: `data/gbm/gbm_data/mirna_gdc/<file UUID>/*.txt` (GDC miRNA quantification; column `reads_per_million_miRNA_mapped`).
Outputs in `data/gbm/gbm_data/`: `gbm_mirna_log2rpm.csv` and `gbm_mirna_zscored.csv` (miRNAs × patients).


In [ ]:
# Repository root: holds config.py (paths and settings) and driver_tiers.py
import os
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# GBM data folder (raw downloads and the files this notebook writes)
CANCER   = "GBM"
prefix   = CANCER.lower()
data_dir = os.path.join(config.BASE_DIR, prefix, f"{prefix}_data")

import pandas as pd
import numpy as np
import requests
from scipy import stats

# Folder of GDC miRNA-seq downloads
mirna_folder = os.path.join(data_dir, "mirna_gdc")


In [ ]:
all_mirna = {}

for uuid in os.listdir(mirna_folder):
    uuid_path = os.path.join(mirna_folder, uuid)
    
    # Skip non-folders (manifest files etc.)
    if not os.path.isdir(uuid_path):
        continue
    
    txt_files = [f for f in os.listdir(uuid_path) if f.endswith(".txt")]
    if not txt_files:
        continue
    
    txt_path = os.path.join(uuid_path, txt_files[0])
    
    try:
        df = pd.read_csv(txt_path, sep="\t", index_col=0)
        
        # Skip metadata files that GDC bundles alongside expression files
        # These have columns like 'submitter_id', 'entity_type' instead of expression values
        if "reads_per_million_miRNA_mapped" not in df.columns:
            continue
        
        # RPM = reads per million miRNA mapped
        # Normalizes for sequencing depth - patients sequenced more deeply
        # would have higher raw counts regardless of biology
        # RPM makes counts comparable across samples
        # Key: folder UUID is the GDC file ID, used later for TCGA barcode mapping
        all_mirna[uuid] = df["reads_per_million_miRNA_mapped"]
        
    except Exception as e:
        print(f"Error in {uuid}: {e}")

print(f"Loaded {len(all_mirna)} samples")

mirna_matrix = pd.DataFrame(all_mirna)

# dropna() keeps only miRNAs detected in ALL 266 samples
# Ensures complete matrix with no missing values for downstream correlation analysis
mirna_matrix = mirna_matrix.dropna()
print(f"Matrix shape: {mirna_matrix.shape}")
# Expected: (1881, 266)


In [ ]:
# Map UUIDs to TCGA barcodes, filter to primary tumor
uuids = mirna_matrix.columns.tolist()
uuid_to_barcode = {}
batch_size = 200

for i in range(0, len(uuids), batch_size):
    batch = uuids[i:i+batch_size]
    payload = {
        "filters": {
            "op": "in",
            "content": {"field": "file_id", "value": batch}
        },
        "fields": "file_id,cases.submitter_id,cases.samples.sample_type",
        "size": len(batch)
    }
    response = requests.post("https://api.gdc.cancer.gov/files", json=payload)
    for hit in response.json()["data"]["hits"]:
        file_id = hit["file_id"]
        case_id = hit["cases"][0]["submitter_id"]
        samples = hit["cases"][0].get("samples", [])
        primary = [s for s in samples if s["sample_type"] == "Primary Tumor"]
        if primary:
            uuid_to_barcode[file_id] = case_id
    print(f"Processed {min(i+batch_size, len(uuids))}/{len(uuids)}")

print(f"Mapped: {len(uuid_to_barcode)} primary tumor samples")

# Filter to primary tumor only then rename
mirna_matrix = mirna_matrix.loc[:, mirna_matrix.columns.isin(uuid_to_barcode.keys())]
mirna_matrix.columns = [uuid_to_barcode[col] for col in mirna_matrix.columns]
print(f"Shape after filtering: {mirna_matrix.shape}")

# Handle duplicates
dupes = mirna_matrix.columns[mirna_matrix.columns.duplicated(keep=False)].unique().tolist()
print(f"Duplicate patients: {len(dupes)}")

if dupes:
    non_dup = mirna_matrix.loc[:, ~mirna_matrix.columns.isin(dupes)]
    best_samples = []
    for patient in dupes:
        cols = mirna_matrix.loc[:, mirna_matrix.columns == patient]
        best = cols.iloc[:, cols.median(axis=0).argmax()]
        best.name = patient
        best_samples.append(best)
    best_dup = pd.concat(best_samples, axis=1)
    mirna_matrix = pd.concat([non_dup, best_dup], axis=1)

print(f"Final shape: {mirna_matrix.shape}")

# log2(RPM+1)
mirna_log = np.log2(mirna_matrix + 1)

# Remove zero variance miRNAs
zero_var = mirna_log.var(axis=1) == 0
print(f"Zero variance miRNAs: {zero_var.sum()}")
mirna_log = mirna_log.loc[~zero_var]

# Z-score
mirna_zscored = pd.DataFrame(
    stats.zscore(mirna_log.values, axis=1),
    index=mirna_log.index,
    columns=mirna_log.columns
)

print(f"\nAfter Z-scoring:")
print(f"Shape: {mirna_zscored.shape}")
print(f"Min: {mirna_zscored.min().min():.2f}")
print(f"Max: {mirna_zscored.max().max():.2f}")
print(f"Mean: {mirna_zscored.mean().mean():.4f}")
print(f"NaN: {mirna_zscored.isna().sum().sum()}")

# Save
mirna_log.to_csv(os.path.join(data_dir, f"{prefix}_mirna_log2rpm.csv"))
mirna_zscored.to_csv(os.path.join(data_dir, f"{prefix}_mirna_zscored.csv"))
print(f"\nSaved {prefix}_mirna_log2rpm.csv and {prefix}_mirna_zscored.csv")
